# Training del Modello di Machine Learning

Questo notebook si occupa della fase di **addestramento e valutazione** del sistema di triage automatico dei ticket.

Presuppone che il preprocessing descritto in `Preprocessing.ipynb` sia già stato eseguito.

I passaggi eseguiti sono:
1. Caricamento e preparazione dei dati
2. Suddivisione Training/Test (80/20)
3. Vettorizzazione TF-IDF
4. Addestramento dei modelli (Regressione Logistica)
5. Valutazione con metriche: Accuracy, Precision, Recall, F1-score
6. Salvataggio dei modelli per la dashboard Streamlit

## 1. Importazione delle librerie

Le librerie utilizzate in questa fase sono:
- **pandas**: per la gestione del dataset
- **scikit-learn**: per TF-IDF, Regressione Logistica e metriche di valutazione
- **pickle**: per salvare i modelli addestrati e riutilizzarli nella dashboard

In [ ]:
import re
import json
import pickle
import unicodedata

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

## 2. Caricamento e preparazione dei dati

Vengono applicati gli stessi passaggi del notebook di preprocessing:
pulizia testuale, rimozione duplicati e unione di oggetto e descrizione.

In [ ]:
def clean_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.lower()
    s = s.replace("\n", " ").replace("\t", " ")
    s = re.sub(r"[^\w\sàèéìòù]", " ", s, flags=re.UNICODE)
    s = re.sub(r"\s+", " ", s).strip()
    return s

df = pd.read_csv("dataset_tickets_pw18.csv")
df["oggetto"] = df["oggetto"].fillna("").apply(clean_text)
df["descrizione"] = df["descrizione"].fillna("").apply(clean_text)
df = df.drop_duplicates(subset=["oggetto", "descrizione"]).reset_index(drop=True)
df["testo"] = (df["oggetto"] + " " + df["descrizione"]).str.strip()

print(f"Dataset pronto: {len(df)} ticket")

## 3. Suddivisione Training/Test

Il dataset viene suddiviso in due parti:
- **Training set (80%)**: usato per addestrare il modello
- **Test set (20%)**: usato per valutare il modello su dati mai visti

Il parametro `random_state=42` garantisce che la suddivisione sia sempre la stessa, rendendo l'esperimento **riproducibile**.

Il parametro `stratify` assicura che la distribuzione delle categorie sia proporzionale in entrambi i set.

In [ ]:
X_train, X_test, y_train_cat, y_test_cat, y_train_prio, y_test_prio = train_test_split(
    df["testo"],
    df["categoria"],
    df["priorita"],
    test_size=0.2,
    random_state=42,
    stratify=df["categoria"]
)

print(f"Training set: {len(X_train)} ticket")
print(f"Test set: {len(X_test)} ticket")

## 4. Vettorizzazione TF-IDF

I modelli di Machine Learning non lavorano direttamente con le parole, ma hanno bisogno di numeri. Per questo motivo viene applicata la tecnica **TF-IDF (Term Frequency - Inverse Document Frequency)**.

TF-IDF assegna un peso a ciascuna parola in base alla sua importanza:
- **TF (Term Frequency)**: quanto spesso una parola appare nel documento
- **IDF (Inverse Document Frequency)**: quanto è rara quella parola nell'intero corpus

Le parole comuni (articoli, preposizioni) vengono penalizzate; quelle rare e significative ottengono un peso maggiore.

⚠️ Il vectorizer viene addestrato **solo sul training set** e poi applicato al test set. Questo evita la cosiddetta *data leakage*, ovvero che il modello veda informazioni del test durante l'addestramento.

In [ ]:
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),  # considera sia parole singole che coppie di parole
    min_df=1,
    max_df=0.95
)

# Fit solo sul training set
X_train_vec = vectorizer.fit_transform(X_train)

# Trasformazione del test set con lo stesso vectorizer
X_test_vec = vectorizer.transform(X_test)

print(f"Dimensioni matrice TF-IDF training: {X_train_vec.shape}")
print(f"Numero di feature (parole/bigrammi): {X_train_vec.shape[1]}")

## 5. Addestramento dei modelli

Vengono addestrati **due modelli distinti**, entrambi basati sulla **Regressione Logistica**:
- `model_cat` → predice la **categoria** del ticket
- `model_prio` → predice la **priorità** del ticket

La Regressione Logistica, nonostante il nome, è un classificatore: calcola la probabilità che un testo appartenga a ciascuna classe e seleziona quella con probabilità più alta.

È stata scelta perché:
- È semplice e interpretabile
- Funziona bene con dataset di dimensioni ridotte
- È veloce da addestrare
- È adatta alla classificazione testuale

Il metodo `fit()` è la funzione che avvia l'addestramento: il modello osserva i dati di training, confronta le sue previsioni con le etichette corrette e aggiusta i propri parametri per ridurre gli errori.

In [ ]:
model_cat = LogisticRegression(max_iter=1000)
model_prio = LogisticRegression(max_iter=1000)

model_cat.fit(X_train_vec, y_train_cat)
model_prio.fit(X_train_vec, y_train_prio)

print("Modelli addestrati con successo!")

## 6. Valutazione del modello — Categoria

La valutazione viene eseguita sul **test set**, ovvero su ticket che il modello non ha mai visto durante l'addestramento.

Le metriche utilizzate sono:
- **Accuracy**: percentuale totale di previsioni corrette
- **Precision**: delle previsioni fatte per una classe, quante sono corrette
- **Recall**: di tutti i ticket di una classe, quanti vengono riconosciuti correttamente
- **F1-score**: media armonica tra Precision e Recall, utile quando le classi non sono bilanciate

In [ ]:
pred_cat = model_cat.predict(X_test_vec)

print("=== RISULTATI — CATEGORIA ===")
print(f"Accuracy: {accuracy_score(y_test_cat, pred_cat):.2%}")
print()
print(classification_report(y_test_cat, pred_cat))

## 7. Valutazione del modello — Priorità

La priorità è generalmente più difficile da classificare rispetto alla categoria, perché è un concetto più soggettivo e le parole che indicano urgenza possono essere ambigue.

In [ ]:
pred_prio = model_prio.predict(X_test_vec)

print("=== RISULTATI — PRIORITÀ ===")
print(f"Accuracy: {accuracy_score(y_test_prio, pred_prio):.2%}")
print()
print(classification_report(y_test_prio, pred_prio))

## 8. Salvataggio dei modelli

I modelli addestrati e il vectorizer vengono salvati su disco in formato `.pkl` tramite la libreria **pickle**.

Questo permette alla dashboard Streamlit (`app.py`) di caricarli direttamente senza dover riaddestrare il modello ogni volta che viene avviata l'applicazione.

In [ ]:
pickle.dump(model_cat, open("model_categoria.pkl", "wb"))
pickle.dump(model_prio, open("model_priorita.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

print("Modelli salvati:")
print("  - model_categoria.pkl")
print("  - model_priorita.pkl")
print("  - vectorizer.pkl")

## 9. Salvataggio report metriche

Le metriche vengono anche salvate in formato JSON per eventuali analisi successive o visualizzazioni grafiche.

In [ ]:
report_cat = classification_report(y_test_cat, pred_cat, output_dict=True)
report_prio = classification_report(y_test_prio, pred_prio, output_dict=True)

with open("report_categoria.json", "w", encoding="utf-8") as f:
    json.dump(report_cat, f, indent=4, ensure_ascii=False)

with open("report_priorita.json", "w", encoding="utf-8") as f:
    json.dump(report_prio, f, indent=4, ensure_ascii=False)

print("Report salvati: report_categoria.json, report_priorita.json")

# Riepilogo finale
print(f"\n=== RIEPILOGO FINALE ===")
print(f"F1-score macro Categoria: {report_cat['macro avg']['f1-score']:.2%}")
print(f"F1-score macro Priorità:  {report_prio['macro avg']['f1-score']:.2%}")

## Riepilogo

Il training è completato. I modelli sono stati:
- ✅ Addestrati con Regressione Logistica su dati TF-IDF
- ✅ Valutati su un test set separato
- ✅ Salvati in formato `.pkl` per l'utilizzo nella dashboard

Il passo successivo è avviare la dashboard interattiva con il comando:
```
streamlit run app.py
```